---
name: enhanceImGuiWidgetStyle
description: Enhance a custom ImGui widget's visual appearance with realistic, physically-inspired styling.
argument-hint: The name or description of the visual effect or styling property to add or improve (e.g., "drop shadow", "rim lighting", "gradient border")
---

Enhance the visual appearance of the custom ImGui widget in the current file to make it look more realistic and physically accurate, without using shaders or render targets — only ImGui's immediate-mode draw list API (`ImDrawList`).

## Requirements

Apply the following improvements where applicable. The user may specify a particular aspect via `$args`; if not specified, apply all relevant enhancements:

### 1. Drop Shadow
- Draw a dark, semi-transparent circle offset slightly toward the bottom-right to simulate Z-axis elevation.
- Optionally support a **soft shadow** mode: use multiple concentric filled circles with a quadratic alpha falloff to simulate a blurred penumbra edge.
- Provide a boolean parameter (e.g., `soft = true`) to toggle between soft and flat shadow.

### 2. 3D Body with Directional Rim Lighting
- Draw the widget body as a filled circle.
- Overlay a gradient rim that simulates a **top-left light source**:
  - Subdivide the full rim into N small arc segments (e.g., 36).
  - For each segment at mid-angle `a`, compute illumination as:
    ```
    illum = -(cos(a) + sin(a)) / sqrt(2)   // range: [-1, +1]
    t     = (illum + 1) / 2                // remap to [0, 1]
    gray  = 255 * t
    alpha = base_alpha + range * t
    ```
  - Render each arc with its computed grayscale color and alpha, producing a smooth continuous gradient rather than a hard split.

### 3. Label `##` Suffix Support (ImGui Convention Compatibility)
- Use `ImGui::FindRenderedTextEnd(label)` as the `text_end` argument to `ImGui::TextUnformatted()` when rendering the label, so that any `##id` suffix is silently stripped from display — consistent with all native ImGui controls.
- Pass the **full** label (including `##id`) to `ImGui::PushID()` so that each widget instance gets a unique ID.
- Pass `hide_text_after_double_hash = true` to `ImGui::CalcTextSize()` so that label centering/sizing ignores the `##` suffix.

## Constraints
- Do **not** use OpenGL/Vulkan shaders, framebuffers, or render targets.
- All rendering must use `ImDrawList` primitives: `AddCircleFilled`, `AddCircle`, `PathArcTo` + `PathStroke`, etc.
- Preserve the existing public API signatures; add new parameters only as **optional** with sensible defaults.
- Ensure all existing call sites continue to compile and behave correctly without modification.
